### Cell 1 — imports + paths + load data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42


In [ ]:
def find_project_root(start: Path) -> Path:
    """Walk upward until we find 'outcomes' folder (your repo marker)."""
    cur = start.resolve()
    for _ in range(12):
        if (cur / "outcomes").exists():
            return cur
        cur = cur.parent
    return start.resolve()

project_root = find_project_root(Path.cwd())
in_fp = project_root / "outcomes" / "s1" / "cleaned_dataset_row2_imputed.csv"
out_dir = project_root / "outcomes" / "s2"
out_dir.mkdir(parents=True, exist_ok=True)

print("project_root:", project_root)
print("input:", in_fp)
print("output dir:", out_dir)
assert in_fp.exists(), f"Input file not found: {in_fp}"

df = pd.read_csv(in_fp)
print("shape:", df.shape)
print("columns:", list(df.columns))


### Cell 3 — define TARGET / ID + significant candidate features (exactly match your column names)

In [ ]:
# Your exact column names (authoritative)
TARGET = "Readmission"
ID_COLS = ["S_no"]

# Significant variables candidates (include all; forward selection will cap at 14)
CANDIDATE_FEATURES = [
    # continuous (significant)
    "Hematocrit",
    "Albumin",
    "Creatinine",
    "MELD Score",

    # binary (significant; 1=Yes, Gender: male=1)
    "Gender",
    "Diabetes",
    "Hypertension",
    "Sleep Apnea",
    "Immunocompromised",
    "PAD",
    "CVD",
    "Prior MI",
    "Pneumonia",
    "Status",

    # categorical (significant; numeric-coded)
    "race",
    "CLD",
    "Num Dis Vessels",
    "Alcohol Use",
    "tobacco_use",
]

# checks
missing = [c for c in [TARGET] + CANDIDATE_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in dataset: {missing}")

print("Using TARGET:", TARGET)
print("Using ID_COLS:", ID_COLS)
print("Candidate features:", len(CANDIDATE_FEATURES))
print(CANDIDATE_FEATURES)


### Cell 4 — build X/y + split train/test

In [ ]:
# drop ID cols if present
drop_cols = [c for c in ID_COLS if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

# X / y
X_all = df[CANDIDATE_FEATURES].copy()
y_all = pd.to_numeric(df[TARGET], errors="coerce").astype(int)

# sanity check y is 0/1
u = sorted(pd.Series(y_all).dropna().unique().tolist())
print("Readmission unique:", u)
assert set(u).issubset({0, 1}), f"Readmission must be 0/1. Found: {u}"

print("Total n:", len(df))
print("Event rate:", float(y_all.mean()))


In [ ]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_all
)

print("Train n:", len(X_train_all), "Test n:", len(X_test_all))
print("Train event rate:", float(y_train.mean()), "Test event rate:", float(y_test.mean()))


### Cell 5 — forward selection setup (CV AUROC)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

base_rf = RandomForestClassifier(
    n_estimators=800,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=5,
)

def cv_auc_for_features(feats):
    Xtr = X_train_all[feats]
    scores = cross_val_score(
        base_rf,
        Xtr,
        y_train,
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1
    )
    return float(scores.mean())


### Cell 6 — forward selection (cap features at 14)

In [ ]:
remaining = list(CANDIDATE_FEATURES)
selected = []
history = []

# target_k = min(14, len(CANDIDATE_FEATURES))  # feature cap = 14
target_k = len(CANDIDATE_FEATURES)

while len(selected) < target_k:
    best_feat = None
    best_score = -np.inf

    for f in remaining:
        trial = selected + [f]
        s = cv_auc_for_features(trial)
        if s > best_score:
            best_score = s
            best_feat = f

    selected.append(best_feat)
    remaining.remove(best_feat)
    history.append((len(selected), best_score, best_feat))
    print(f"step {len(selected):02d}: add {best_feat} -> CV AUROC = {best_score:.4f}")


### Cell 7 — pick best k in 1..14 by CV AUROC + save history

In [ ]:
best_k = None
best_k_score = -np.inf
best_feats = None

for k in range(1, target_k + 1):
    feats_k = selected[:k]
    s = cv_auc_for_features(feats_k)
    print(f"k={k:02d} CV AUROC={s:.4f} feats={feats_k}")
    if s > best_k_score:
        best_k_score = s
        best_k = k
        best_feats = feats_k

BEST_FEATURES = best_feats

print("\nBest k =", best_k)
print("Best CV AUROC =", round(best_k_score, 4))
print("Selected features:", BEST_FEATURES)

hist_df = pd.DataFrame(history, columns=["k", "cv_auc_at_step", "added_feature"])
hist_df.to_csv(out_dir / "rf_forward_selection_history.csv", index=False)
print("Saved:", out_dir / "rf_forward_selection_history.csv")


### Testing 1.1 for different splits

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

RANDOM_STATE_BASE = 42

# ====== Use the previously identified BEST_FEATURES ======
# X_all and y_all were defined earlier (df + CANDIDATE_FEATURES / TARGET).
X_all = df[BEST_FEATURES].copy()
y_all = pd.to_numeric(df["Readmission"], errors="coerce").astype(int)  # Target column: "Readmission"

# ====== Previously selected best parameters ======
BASE_PARAMS = {
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "max_depth": None,
    "max_features": "sqrt",
    "min_impurity_decrease": 1e-4,
    "min_samples_leaf": 20,
    "min_samples_split": 2,
    "n_estimators": 1000,
    "random_state": 42,
    "n_jobs": -1,
}

# ====== Narrow search around the best parameters (much faster) ======
# Sample with RandomizedSearchCV instead of evaluating all 2,592 combinations.
param_dist = {
    "n_estimators": [800, 1000, 1200],
    "min_samples_leaf": [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_impurity_decrease": [0.0, 1e-4, 3e-4],
    "max_features": ["sqrt", 0.5, 0.8],
    "max_depth": [None, 8, 12],
    "bootstrap": [True],
    "class_weight": ["balanced_subsample"],
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE_BASE)


### Testing 1.2: Find the best within-split CV AUROC across multiple train/test splits

In [ ]:
from time import time

def eval_one_split(seed: int, test_size: float = 0.25, n_iter: int = 30):
    """
    For each split seed:
      - Partition the data into training and test sets.
      - Use RandomizedSearchCV on the training set for a narrow search (n_iter samples).
      - Return best_cv_auc, best_params, and test_auc.
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=seed,
        stratify=y_all
    )

    rf = RandomForestClassifier(**BASE_PARAMS)

    rs = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=n_iter,                 # Keep this small for speed (e.g., 20–40).
        scoring="roc_auc",
        cv=cv5,
        n_jobs=-1,
        refit=True,
        random_state=seed,
        verbose=0
    )

    t0 = time()
    rs.fit(X_tr, y_tr)
    fit_sec = time() - t0

    best_cv = float(rs.best_score_)
    best_est = rs.best_estimator_
    p_te = best_est.predict_proba(X_te)[:, 1]
    test_auc = float(roc_auc_score(y_te, p_te))

    return {
        "split_seed": seed,
        "n_train": len(y_tr),
        "n_test": len(y_te),
        "best_cv_auc": best_cv,
        "test_auc": test_auc,
        "best_params": rs.best_params_,
        "fit_seconds": fit_sec,
    }

# ====== Number of splits to evaluate ======
SPLIT_SEEDS = list(range(100, 120))  # Evaluate 20 splits; adjust this range as needed.
N_ITER = 30                          # Sample 30 parameter combinations per split.

results = []
for seed in SPLIT_SEEDS:
    r = eval_one_split(seed=seed, test_size=0.25, n_iter=N_ITER)
    results.append(r)
    print(f"seed={seed} best_cv_auc={r['best_cv_auc']:.4f} test_auc={r['test_auc']:.4f} time={r['fit_seconds']:.1f}s")

res_df = pd.DataFrame(results).sort_values("best_cv_auc", ascending=False)
res_df.head(10)


### Testing 1.3: Select the split with the highest best CV AUROC and save its results to s2

In [ ]:
# Select the split with the highest CV AUROC.
best_row = res_df.iloc[0].to_dict()

print("\n=== BEST SPLIT BY CV AUROC ===")
print("split_seed:", best_row["split_seed"])
print("best_cv_auc:", round(best_row["best_cv_auc"], 6))
print("test_auc:", round(best_row["test_auc"], 6))
print("best_params:", best_row["best_params"])

# Save the summary table (best_params can remain a dictionary).
out_csv = out_dir / "rf_split_sensitivity_fastsearch.csv"
res_df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)

# Save the optimal parameter dictionary separately as a TXT file for reuse.
best_txt = out_dir / "rf_best_split_and_params.txt"
with open(best_txt, "w", encoding="utf-8") as f:
    f.write(f"BEST_FEATURES={BEST_FEATURES}\n")
    f.write(f"best_split_seed={best_row['split_seed']}\n")
    f.write(f"best_cv_auc={best_row['best_cv_auc']}\n")
    f.write(f"test_auc={best_row['test_auc']}\n")
    f.write(f"best_params={best_row['best_params']}\n")
print("Saved:", best_txt)


### Testing 1.4 (optional but recommended): Guard against high CV AUROC with poor test performance

In [ ]:
print("\n=== TOP 10 BY TEST AUC ===")
print(res_df.sort_values("test_auc", ascending=False)[
    ["split_seed", "best_cv_auc", "test_auc", "fit_seconds"]
].head(10).to_string(index=False))

# A simple robustness check: among the five best CV splits, choose the highest test AUROC.
topk = 5
cand = res_df.head(topk).copy()
cand_best = cand.sort_values("test_auc", ascending=False).iloc[0]

print(f"\n=== RECOMMENDED (among top {topk} CV, best TEST) ===")
print("split_seed:", int(cand_best["split_seed"]))
print("best_cv_auc:", round(float(cand_best["best_cv_auc"]), 6))
print("test_auc:", round(float(cand_best["test_auc"]), 6))
print("best_params:", cand_best["best_params"])


### Testing 2.1: Fix the random seed and scan test_size across different splits

In [ ]:
import numpy as np
import pandas as pd
from time import time

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# ----------------------------
# (A) Inputs
# ----------------------------
SAME_SEED = 42
TARGET = "Readmission"

BEST_FEATURES = [
    "Albumin", "Alcohol Use", "Hypertension", "Immunocompromised", "CVD",
    "Num Dis Vessels", "Prior MI", "Hematocrit", "PAD", "Sleep Apnea",
    "Creatinine", "Gender", "Pneumonia", "tobacco_use"
]

# fixed best params (NO retuning)
FIXED_RF_PARAMS = dict(
    bootstrap=True,
    class_weight="balanced_subsample",
    max_depth=None,
    max_features="sqrt",
    min_impurity_decrease=1e-4,
    min_samples_leaf=20,
    min_samples_split=2,
    n_estimators=1000,
    random_state=SAME_SEED,
    n_jobs=-1,
)

TEST_SIZES = [0.15, 0.20, 0.25, 0.30, 0.35]  # change if you want
CV_FOLDS = 5

# ----------------------------
# (B) checks
# ----------------------------
missing = [c for c in [TARGET] + BEST_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in df: {missing}")

X_all = df[BEST_FEATURES].copy()
y_all = pd.to_numeric(df[TARGET], errors="coerce").astype(int)

print("Total n:", len(df))
print("Event rate:", float(y_all.mean()))
print("Using fixed RF params:", {k: FIXED_RF_PARAMS[k] for k in FIXED_RF_PARAMS if k != "n_jobs"})
print("Using features (k={}):".format(len(BEST_FEATURES)), BEST_FEATURES)

# fixed 5-fold CV splitter (same each time; only data changes with split)
cv5 = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SAME_SEED)

rf_fixed = RandomForestClassifier(**FIXED_RF_PARAMS)

# ----------------------------
# (C) Evaluate each split
# ----------------------------
rows = []
for ts in TEST_SIZES:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all,
        test_size=ts,
        stratify=y_all,
        random_state=SAME_SEED,
        shuffle=True
    )

    t0 = time()
    cv_scores = cross_val_score(
        rf_fixed,
        X_tr, y_tr,
        scoring="roc_auc",
        cv=cv5,
        n_jobs=-1
    )
    sec = time() - t0

    # optional: fit once and report test AUROC
    rf_fixed.fit(X_tr, y_tr)
    p_te = rf_fixed.predict_proba(X_te)[:, 1]
    test_auc = float(roc_auc_score(y_te, p_te))

    rows.append({
        "test_size": ts,
        "n_train": int(len(y_tr)),
        "n_test": int(len(y_te)),
        "cv_auc_mean": float(np.mean(cv_scores)),
        "cv_auc_std": float(np.std(cv_scores, ddof=1)),
        "test_auc": test_auc,
        "seconds_cv_only": sec
    })

    print(
        f"test_size={ts:.2f}  "
        f"CV AUROC={np.mean(cv_scores):.4f}±{np.std(cv_scores, ddof=1):.4f}  "
        f"test AUROC={test_auc:.4f}  "
        f"time={sec:.1f}s"
    )

res_df = pd.DataFrame(rows).sort_values("cv_auc_mean", ascending=False)

print("\n=== Best split by fixed-params CV AUROC ===")
best = res_df.iloc[0]
print("test_size:", best["test_size"])
print("cv_auc_mean:", round(best["cv_auc_mean"], 6))
print("cv_auc_std:", round(best["cv_auc_std"], 6))
print("test_auc:", round(best["test_auc"], 6))

# save
out_csv = out_dir / "rf_fixed_params_same_seed_vary_test_size.csv"
res_df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)

res_df


### Testing 1.3: optimal seed (102, 115) and optimal splits (0.25)

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# USER SETTINGS
# -----------------------------
TARGET = "Readmission"   # Actual target column name.
SEEDS = [101, 102, 103, 114, 115, 116]
TEST_SIZES = [0.20, 0.25, 0.30]

# Do not set random_state here; it is passed separately to the model below.
FIXED_RF_PARAMS = dict(
    bootstrap=True,
    class_weight="balanced_subsample",
    max_depth=None,
    max_features="sqrt",
    min_impurity_decrease=1e-4,
    min_samples_leaf=20,
    min_samples_split=2,
    n_estimators=1000,
    n_jobs=-1,
)

# sanity checks
missing_cols = [c for c in [TARGET] + list(BEST_FEATURES) if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

X_all = df[BEST_FEATURES].copy()
y_all = df[TARGET].astype(int).copy()

print("Total n:", len(df))
print("Event rate:", float(y_all.mean()))
print("Using features (k=%d):" % len(BEST_FEATURES), BEST_FEATURES)
print("Using fixed RF params:", FIXED_RF_PARAMS)

# CV setup (train-only CV)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_one(seed: int, test_size: float):
    # split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all,
        test_size=test_size,
        random_state=seed,
        stratify=y_all
    )

    # model (seed controls the RF randomness for this run)
    rf = RandomForestClassifier(
        random_state=seed,
        **FIXED_RF_PARAMS
    )

    # train-CV AUROC on training set only
    cv_scores = cross_val_score(
        rf, X_tr, y_tr,
        scoring="roc_auc",
        cv=CV,
        n_jobs=-1
    )

    # fit on full train, test AUROC
    rf.fit(X_tr, y_tr)
    p_te = rf.predict_proba(X_te)[:, 1]
    test_auc = roc_auc_score(y_te, p_te)

    return float(cv_scores.mean()), float(cv_scores.std(ddof=1)), float(test_auc), len(y_te)

# -----------------------------
# GRID SEARCH OVER (seed, split)
# -----------------------------
rows = []
t0 = time.time()

for ts in TEST_SIZES:
    for seed in SEEDS:
        cv_mean, cv_sd, test_auc, n_test = eval_one(seed, ts)
        rows.append({
            "seed": seed,
            "test_size": ts,
            "n_test": n_test,
            "cv_auc_mean": cv_mean,
            "cv_auc_sd": cv_sd,
            "test_auc": test_auc
        })
        print(f"seed={seed} test_size={ts:.2f}  CV={cv_mean:.4f}±{cv_sd:.4f}  Test={test_auc:.4f}")

res = pd.DataFrame(rows)

print("\nDone. Total time: %.1fs" % (time.time() - t0))

# -----------------------------
# PICK BEST COMBOS
# -----------------------------
best_by_cv = res.sort_values(["cv_auc_mean", "test_auc"], ascending=[False, False]).head(1)
best_by_test = res.sort_values(["test_auc", "cv_auc_mean"], ascending=[False, False]).head(1)

print("\n=== Best combo by TRAIN CV AUROC ===")
print(best_by_cv.to_string(index=False))

print("\n=== Best combo by TEST AUROC (FYI) ===")
print(best_by_test.to_string(index=False))

# save
out_csv = out_dir / "rf_seed_split_grid_fixed_params.csv"
res.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)


### Cell 8 — GridSearchCV on BEST_FEATURES (optimize CV AUROC)

In [ ]:
Xtr = X_train_all[BEST_FEATURES].copy()
Xte = X_test_all[BEST_FEATURES].copy()

rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [600, 1000, 1500],
    "max_depth": [None, 4, 6, 8],
    "min_samples_leaf": [5, 10, 20, 40],
    "min_samples_split": [2, 10, 30],
    "max_features": ["sqrt", 0.5, 0.8],
    "class_weight": ["balanced_subsample", "balanced"],
    "bootstrap": [True],
    "min_impurity_decrease": [0.0, 1e-4, 1e-3],
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gs_auc = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv5,
    n_jobs=-1,
    refit=True,
    verbose=1
)

gs_auc.fit(Xtr, y_train)

print("\nBest CV AUROC:", gs_auc.best_score_)
print("Best params:", gs_auc.best_params_)


### Cell 9 — test AUROC + save summary to outcomes/s2

In [ ]:
best_rf = gs_auc.best_estimator_
p_te = best_rf.predict_proba(Xte)[:, 1]
test_auc = roc_auc_score(y_test, p_te)

print("Test AUROC:", test_auc)

summary = pd.DataFrame({
    "best_k": [best_k],
    "best_cv_auc_forward_selection": [best_k_score],
    "best_cv_auc_gridsearch": [gs_auc.best_score_],
    "test_auc": [test_auc],
    "best_features": [",".join(BEST_FEATURES)],
    "best_params": [str(gs_auc.best_params_)]
})

summary_fp = out_dir / "rf_best_model_summary.csv"
summary.to_csv(summary_fp, index=False)

print("Saved:", summary_fp)


## Applying trained model into train dataset for optimal AUC.

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    """Walk upward until we find 'outcomes' folder (repo marker)."""
    cur = start.resolve()
    for _ in range(12):
        if (cur / "outcomes").exists():
            return cur
        cur = cur.parent
    return start.resolve()

project_root = find_project_root(Path.cwd())
in_fp = project_root / "outcomes" / "s1" / "cleaned_dataset_row2_imputed.csv"
out_dir = project_root / "outcomes" / "s2"
out_dir.mkdir(parents=True, exist_ok=True)

print("project_root:", project_root)
print("input:", in_fp)
print("output dir:", out_dir)

assert in_fp.exists(), f"Input file not found: {in_fp}"

df = pd.read_csv(in_fp)

print("shape:", df.shape)
print("columns:", list(df.columns))


In [ ]:
# Cell 1 — use the best combo you found (seed=103, test_size=0.20) to define train/test
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

TARGET = "Readmission"
BEST_SEED = 103
BEST_TEST_SIZE = 0.20

BEST_FEATURES = [
    "Albumin", "Alcohol Use", "Hypertension", "Immunocompromised", "CVD",
    "Num Dis Vessels", "Prior MI", "Hematocrit", "PAD", "Sleep Apnea",
    "Creatinine", "Gender", "Pneumonia", "tobacco_use"
]

FIXED_RF_PARAMS = dict(
    bootstrap=True,
    class_weight="balanced_subsample",
    max_depth=None,
    max_features="sqrt",
    min_impurity_decrease=1e-4,
    min_samples_leaf=20,
    min_samples_split=2,
    n_estimators=1000,
    n_jobs=-1,
)

# sanity
need = [TARGET] + BEST_FEATURES
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in df: {missing}")

X_all = df[BEST_FEATURES].copy()
y_all = df[TARGET].astype(int).copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all,
    test_size=BEST_TEST_SIZE,
    random_state=BEST_SEED,
    stratify=y_all
)

print("Total n:", len(df))
print("Train n:", len(y_tr), "Test n:", len(y_te))
print("Train event rate:", float(y_tr.mean()), "Test event rate:", float(y_te.mean()))


In [ ]:
# Cell 2 — CV on TRAIN ONLY using the optimal model settings
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_best = RandomForestClassifier(
    random_state=BEST_SEED,   # controls RF randomness
    **FIXED_RF_PARAMS
)

cv_scores = cross_val_score(
    rf_best, X_tr, y_tr,
    scoring="roc_auc",
    cv=CV,
    n_jobs=-1
)

print("Train 5-fold CV AUROC (mean ± sd): %.4f ± %.4f" % (cv_scores.mean(), cv_scores.std(ddof=1)))
print("Fold scores:", np.round(cv_scores, 4))


In [ ]:
# Cell 3 — fit on full TRAIN, evaluate TEST AUROC (to match your grid output)
rf_best.fit(X_tr, y_tr)
p_te = rf_best.predict_proba(X_te)[:, 1]
test_auc = roc_auc_score(y_te, p_te)

print("Test AUROC:", round(float(test_auc), 4))


In [ ]:
# Cell 4 — (optional) save the CV results for reporting
out_csv = out_dir / "rf_best_combo_train_cv_only.csv"
pd.DataFrame({
    "best_seed": [BEST_SEED]*len(cv_scores),
    "best_test_size": [BEST_TEST_SIZE]*len(cv_scores),
    "fold": np.arange(1, len(cv_scores)+1),
    "train_cv_auc": cv_scores
}).to_csv(out_csv, index=False)

print("Saved:", out_csv)


### Using best model of RF from train dataset to obtain the AUC from train dataset.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# assume these already exist
# -----------------------------
# df               : full dataframe
# BEST_FEATURES    : 14 selected features
# TARGET           : "Readmission"
# seed             : 103
# test_size        : 0.20
# FIXED_RF_PARAMS  : your tuned params (NO random_state inside)

# -----------------------------
# train / test split (same as before)
# -----------------------------
from sklearn.model_selection import train_test_split

X_all = df[BEST_FEATURES].copy()
y_all = df[TARGET].astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all,
    test_size=0.20,
    random_state=103,
    stratify=y_all
)

print("Train n:", len(X_tr))
print("Test n :", len(X_te))
print("Train event rate:", float(y_tr.mean()))

# -----------------------------
# STEP 1: fit on FULL train
# -----------------------------
best_rf = RandomForestClassifier(
    random_state=103,
    **FIXED_RF_PARAMS
)

best_rf.fit(X_tr, y_tr)

# -----------------------------
# STEP 2: leaky CV on train
# -----------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores_leaky = cross_val_score(
    best_rf,          # 🚨 already-fitted model
    X_tr,
    y_tr,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

print("\n❌ LEAKY train CV AUROC")
print("Fold AUCs:", np.round(cv_scores_leaky, 4))
print(
    "Mean ± SD:",
    f"{cv_scores_leaky.mean():.4f} ± {cv_scores_leaky.std(ddof=1):.4f}"
)


In [ ]:
from sklearn.metrics import roc_auc_score

# fit on full train
best_rf.fit(X_tr, y_tr)

# predict on the SAME train set
p_tr = best_rf.predict_proba(X_tr)[:, 1]
train_auc = roc_auc_score(y_tr, p_tr)

print("❌ Train AUC (evaluate on same train used to fit):", round(train_auc, 4))


### Used one 1/29 6:23 PM

In [ ]:
# =======================
# TABLE 2 (FULL VERSION)
# Adds: Logistic Regression, SVM, Neural Network (MLP)
# - Trains these 3 models on the SAME (X_tr, y_tr) and evaluates:
#   (1) train-leaky (fit on train, eval on same train)
#   (2) test (fit on train, eval on test)
# - Keeps your RF exactly optimal: seed=103 + FIXED_RF_PARAMS
# - Tunes XGB quickly (train CV AUROC) so it’s not too bad but still fast
#
# Output:
# - Table2_train_leaky.csv
# - Table2_test.csv
# =======================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, recall_score, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from sklearn.model_selection import ParameterGrid

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# -----------------------
# 0) Sanity
# -----------------------
need_globals = ["X_tr", "y_tr", "X_te", "y_te", "FIXED_RF_PARAMS", "BEST_SEED", "out_dir"]
missing_g = [g for g in need_globals if g not in globals()]
assert not missing_g, f"Missing objects from previous cells: {missing_g}"

seed = int(BEST_SEED)

# -----------------------
# 1) Metrics helper
# -----------------------
def eval_metrics(y_true, proba, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba).astype(float)
    pred = (proba >= thr).astype(int)
    return {
        "AUC": float(roc_auc_score(y_true, proba)),
        "PRAUC": float(average_precision_score(y_true, proba)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "Accuracy": float(accuracy_score(y_true, pred)),
    }

# -----------------------
# 2) RF EXACTLY as your reference (seed + FIXED_RF_PARAMS)
# -----------------------
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_for_cv = RandomForestClassifier(
    random_state=seed,
    **FIXED_RF_PARAMS
)

cv_aucs = cross_val_score(
    rf_for_cv,
    X_tr, y_tr,
    scoring="roc_auc",
    cv=cv5,
    n_jobs=-1
)

print("RF Train 5CV AUROC folds:", np.round(cv_aucs, 4))
print("RF Train 5CV AUROC mean±sd:", f"{cv_aucs.mean():.4f} ± {cv_aucs.std(ddof=1):.4f}")

best_rf = clone(rf_for_cv)
best_rf.fit(X_tr, y_tr)

p_tr_rf = best_rf.predict_proba(X_tr)[:, 1]
p_te_rf = best_rf.predict_proba(X_te)[:, 1]

rf_train_leaky = eval_metrics(y_tr, p_tr_rf, thr=0.5)
rf_test = eval_metrics(y_te, p_te_rf, thr=0.5)

print("RF Train AUC (fit+eval on same train):", round(rf_train_leaky["AUC"], 4))
print("RF Test AUROC:", round(rf_test["AUC"], 4))

# -----------------------
# 3) XGB for Table 2 (fast, not-too-bad) - tune by TRAIN CV AUROC
# -----------------------
pos = int(np.sum(np.asarray(y_tr) == 1))
neg = int(np.sum(np.asarray(y_tr) == 0))
scale_pos_weight = (neg / max(pos, 1))

xgb_grid = {
    "n_estimators": [300, 600, 900],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.06],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda": [1.0, 5.0],
}

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_xgb_params = None
best_xgb_cv = -np.inf

t0 = time.time()
for params in ParameterGrid(xgb_grid):
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        random_state=seed,
        scale_pos_weight=scale_pos_weight,
        **params
    )
    scores = cross_val_score(model, X_tr, y_tr, scoring="roc_auc", cv=CV, n_jobs=-1)
    m = float(np.mean(scores))
    if m > best_xgb_cv:
        best_xgb_cv = m
        best_xgb_params = params

print("\nBest XGB train-CV AUROC:", round(best_xgb_cv, 4))
print("Best XGB params:", best_xgb_params)
print("XGB tuning time(s):", round(time.time() - t0, 1))

best_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=seed,
    scale_pos_weight=scale_pos_weight,
    **best_xgb_params
)
best_xgb.fit(X_tr, y_tr)

# -----------------------
# 4) Add 3 baseline models (train them now)
#    - Logistic Regression (scaled)
#    - SVM (probability=True, scaled)
#    - Neural Network / MLP (scaled)
# -----------------------
# Logistic Regression
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        solver="lbfgs",
        max_iter=3000,
        class_weight="balanced",
        random_state=seed
    ))
])

# SVM (RBF) - keep modest so it doesn’t take forever
svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=seed
    ))
])

# Neural Network (MLP)
mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-3,
        learning_rate_init=1e-3,
        max_iter=400,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=seed
    ))
])

print("\nTraining Logistic regression / SVM / Neural Network (MLP)...")
t1 = time.time()
logreg.fit(X_tr, y_tr)
svm.fit(X_tr, y_tr)
mlp.fit(X_tr, y_tr)
print("Done. time(s):", round(time.time() - t1, 1))

# -----------------------
# 5) Build model dict in the exact order you want on vertical axis
# -----------------------
models = {
    "RF": best_rf,
    "XGB": best_xgb,
    "SVM": svm,
    "Neural Network": mlp,
    "Logistic regression": logreg,
}

# -----------------------
# 6) Make Table 2 (train-leaky + test)
# -----------------------
def make_table2(models: dict, X_tr, y_tr, X_te, y_te, thr=0.5):
    rows_train, rows_test = [], []

    for name, model in models.items():
        # train-leaky
        p_tr = model.predict_proba(X_tr)[:, 1]
        mt = eval_metrics(y_tr, p_tr, thr=thr)
        rows_train.append({"Model": name, **mt})

        # test
        p_te = model.predict_proba(X_te)[:, 1]
        me = eval_metrics(y_te, p_te, thr=thr)
        rows_test.append({"Model": name, **me})

    df_train = pd.DataFrame(rows_train)
    df_test  = pd.DataFrame(rows_test)

    cols = ["Model", "AUC", "PRAUC", "F1", "Recall", "Accuracy"]
    df_train = df_train[cols].reset_index(drop=True)
    df_test  = df_test[cols].reset_index(drop=True)

    return df_train, df_test

t2_train, t2_test = make_table2(models, X_tr, y_tr, X_te, y_te, thr=0.5)

# -----------------------
# 7) Save + display
# -----------------------
t2_train_path = out_dir / "Table2_train_leaky.csv"
t2_test_path  = out_dir / "Table2_test.csv"

t2_train.to_csv(t2_train_path, index=False)
t2_test.to_csv(t2_test_path, index=False)

print("\nSaved:", t2_train_path)
print("Saved:", t2_test_path)

print("\nTable 2 (train-leaky):")
display(t2_train.style.format({c: "{:.4f}" for c in ["AUC","PRAUC","F1","Recall","Accuracy"]}))

print("\nTable 2 (test):")
display(t2_test.style.format({c: "{:.4f}" for c in ["AUC","PRAUC","F1","Recall","Accuracy"]}))
